In [1]:
# ================================
# AI6130 Open-LLM Benchmark Robust Script
# ================================

!pip install -q transformers datasets accelerate tqdm pandas

import datasets
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import os

# -------------------------------
# Suppress auto-conversion warnings
# -------------------------------
os.environ["TRANSFORMERS_NO_AUTO_CONVERT"] = "1"

# -------------------------------
# 1️⃣ Load dataset and group by key
# -------------------------------
print("Loading Open-LLM Benchmark dataset...")
dataset = datasets.load_dataset("Open-Style/Open-LLM-Benchmark", "questions")

grouped = {}
for ex in dataset["train"]:
    d = ex["dataset"]
    if d not in grouped:
        grouped[d] = []
    grouped[d].append(ex)

print("All dataset keys detected:")
print(list(grouped.keys()))

# -------------------------------
# 2️⃣ Map dataset keys to tasks
# -------------------------------
task_data = {}
for key in grouped.keys():
    lower_key = key.lower()
    if "commonsense" in lower_key:
        task_data["CommonsenseQA"] = grouped[key]
    if "openbook" in lower_key:
        task_data["OpenbookQA"] = grouped[key]
    if "piqa" in lower_key:
        task_data["PiQA"] = grouped[key]

print("Mapped tasks:", task_data.keys())
tasks = list(task_data.keys())

# -------------------------------
# 3️⃣ Models to evaluate
# -------------------------------
models = {
    "TinyLlama": "TinyLlama/TinyLlama_v1.1",
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "DeepSeek-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "Qwen3-4B": "Qwen/Qwen3-4B-Instruct-2507"
}

# -------------------------------
# 4️⃣ Best Prompt (Few-shot + reasoning + structured answer)
# -------------------------------
def build_prompt(sample):
    question = sample["question"]
    options = sample["options"]

    prompt = """
You are a helpful AI that solves multiple-choice reasoning questions.

Think carefully before answering.

Example 1
Question: Where do people usually sleep?
A. kitchen
B. bedroom
C. garage
D. garden
Reasoning: People usually sleep in bedrooms where beds are located.
Final Answer: B

Example 2
Question: What helps plants grow?
A. sunlight
B. plastic
C. metal
D. sand
Reasoning: Plants need sunlight for photosynthesis.
Final Answer: A

Now answer the following.

Question:
"""
    prompt += question + "\n\nChoices:\n"
    for option in options:
        prompt += f"{option['label']}. {option['text']}\n"
    prompt += "\nReasoning:\nExplain briefly.\nThen output the final answer in this format:\nFinal Answer: A/B/C/D\n"
    return prompt

# -------------------------------
# 5️⃣ Robust Answer Extraction
# -------------------------------
def extract_answer(text):
    """
    Extract final answer from model output.
    Returns single uppercase letter or 'X' if missing.
    """
    text = text.strip()
    if not text:
        return "X"

    if "Final Answer:" in text:
        ans = text.split("Final Answer:")[-1].strip()
    elif "Answer:" in text:
        ans = text.split("Answer:")[-1].strip()
    else:
        ans = text

    if len(ans) == 0:
        return "X"
    first_char = ans[0].upper()
    if first_char in ["A", "B", "C", "D"]:
        return first_char
    else:
        return "X"

# -------------------------------
# 6️⃣ Inference
# -------------------------------
def infer(sample, tokenizer, model):
    prompt = build_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,  # increased for reasoning
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text, extract_answer(text)

# -------------------------------
# 7️⃣ Evaluation
# -------------------------------
def evaluate(samples, tokenizer, model, examples_per_task=2):
    correct = 0
    reasoning_examples = []
    for idx, sample in enumerate(tqdm(samples)):
        text, pred = infer(sample, tokenizer, model)
        if pred == sample["answerKey"].upper():
            correct += 1
        elif pred == "X":
            print("Warning: empty/invalid prediction for question:", sample["question"])

        # collect 1-2 examples per task
        if len(reasoning_examples) < examples_per_task:
            reasoning_examples.append({
                "question": sample["question"],
                "prediction": pred,
                "answerKey": sample["answerKey"].upper(),
                "reasoning_output": text
            })
    accuracy = correct / len(samples) * 100
    return accuracy, reasoning_examples

# -------------------------------
# 8️⃣ Benchmark Loop
# -------------------------------
results = {}
all_reasoning_examples = {}

for label, model_name in models.items():
    print("\n============================")
    print("Loading Model:", label)
    print("============================")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )

    results[label] = {}
    all_reasoning_examples[label] = {}

    for task in tasks:
        print("\nEvaluating:", task)
        samples = task_data[task][:200]  # limit for speed
        start = time.time()
        acc, reasoning_examples = evaluate(samples, tokenizer, model)
        end = time.time()
        print("Accuracy:", acc, "| Time:", round(end-start,2), "s")
        results[label][task] = acc
        all_reasoning_examples[label][task] = reasoning_examples

    del model
    torch.cuda.empty_cache()

# -------------------------------
# 9️⃣ Results Table
# -------------------------------
table = pd.DataFrame(results).T
print("\nFinal Benchmark Results:")
display(table)
table.to_csv("benchmark_results.csv")
print("\nSaved results to 'benchmark_results.csv'")

# -------------------------------
# 10️⃣ Display 1-2 reasoning examples per task per model
# -------------------------------
for model_name in all_reasoning_examples:
    for task_name in all_reasoning_examples[model_name]:
        print(f"\n--- {model_name} | {task_name} | Sample Reasoning ---")
        for ex in all_reasoning_examples[model_name][task_name]:
            print("Q:", ex["question"])
            print("Predicted:", ex["prediction"], "| True:", ex["answerKey"])
            print("Model Reasoning:\n", ex["reasoning_output"])
            print("-"*50)

Loading Open-LLM Benchmark dataset...


README.md: 0.00B [00:00, ?B/s]

ARC.json: 0.00B [00:00, ?B/s]

CommonsenseQA.json: 0.00B [00:00, ?B/s]

Hellaswag.json: 0.00B [00:00, ?B/s]

...

 86%|████████▌ | 172/200 [13:55<02:16,  4.88s/it]

100%|██████████| 200/200 [16:14<00:00,  4.87s/it]

Accuracy: 65.5 | Time: 974.25 s

Final Benchmark Results:


,CommonsenseQA,OpenbookQA,PiQA
TinyLlama,18.5,27.0,50.0
Qwen2.5-3B,24.5,28.0,53.0
DeepSeek-1.5B,18.5,15.0,38.0
Qwen3-4B,37.5,61.5,65.5



Saved results to 'benchmark_results.csv'

--- TinyLlama | CommonsenseQA | Sample Reasoning ---
Q: A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?
Predicted: A | True: A
Model Reasoning:
 
You are a helpful AI that solves multiple-choice reasoning questions.

Think carefully before answering.

Example 1
Question: Where do people usually sleep?
A. kitchen
B. bedroom
C. garage
D. garden
Reasoning: People usually sleep in bedrooms where beds are located.
Final Answer: B

Example 2
Question: What helps plants grow?
A. sunlight
B. plastic
C. metal
D. sand
Reasoning: Plants need sunlight for photosynthesis.
Final Answer: A

Now answer the following.

Question:
A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?

Choices:
A. bank
B. library
C. department store
D. mall
E. new york

Reasoning:
Explain briefly.
Then output the final answer in this format:
Final Answer: A/B/C/D

Q

In [1]:
# ================================
# AI6130 Open-LLM Benchmark - Original vs Improved Prompt Comparison
# ================================

!pip install -q transformers datasets accelerate tqdm pandas

import datasets
import torch
import time
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
import pandas as pd
import os
import numpy as np

# -------------------------------
# Suppress auto-conversion warnings
# -------------------------------
os.environ["TRANSFORMERS_NO_AUTO_CONVERT"] = "1"

# -------------------------------
# 1️⃣ Load dataset and group by task
# -------------------------------
print("Loading Open-LLM Benchmark dataset...")
dataset = datasets.load_dataset("Open-Style/Open-LLM-Benchmark", "questions")

grouped = {}
for ex in dataset["train"]:
    d = ex["dataset"]
    if d not in grouped:
        grouped[d] = []
    grouped[d].append(ex)

print("All dataset keys detected:")
print(list(grouped.keys()))

# -------------------------------
# 2️⃣ Map dataset keys to required tasks
# -------------------------------
task_data = {}
for key in grouped.keys():
    lower_key = key.lower()
    if "commonsense" in lower_key:
        task_data["CommonsenseQA"] = grouped[key]
    if "openbook" in lower_key:
        task_data["OpenbookQA"] = grouped[key]
    if "piqa" in lower_key:
        task_data["PiQA"] = grouped[key]

print("Mapped tasks:", task_data.keys())
tasks = list(task_data.keys())
NUM_SAMPLES = 200  # limit for speed (set to None for full)

# -------------------------------
# 3️⃣ Models to evaluate
# -------------------------------
models = {
    "TinyLlama": "TinyLlama/TinyLlama_v1.1",
    "Qwen2.5-3B": "Qwen/Qwen2.5-3B-Instruct",
    "DeepSeek-1.5B": "deepseek-ai/DeepSeek-R1-Distill-Qwen-1.5B",
    "Qwen3-4B": "Qwen/Qwen3-4B-Instruct-2507"
}

# -------------------------------
# 4️⃣ Prompt builders
# -------------------------------
def build_prompt_original(sample):
    """Simple prompt: question + options + 'Answer:'"""
    question = sample["question"]
    options = sample["options"]
    prompt = f"{question}\n"
    for opt in options:
        prompt += f"{opt['label']}. {opt['text']}\n"
    prompt += "Answer:"
    return prompt

def build_prompt_improved(sample):
    """Few-shot with reasoning instruction (your best prompt)"""
    question = sample["question"]
    options = sample["options"]

    prompt = """
You are a helpful AI that solves multiple-choice reasoning questions.

Think carefully before answering.

Example 1
Question: Where do people usually sleep?
A. kitchen
B. bedroom
C. garage
D. garden
Reasoning: People usually sleep in bedrooms where beds are located.
Final Answer: B

Example 2
Question: What helps plants grow?
A. sunlight
B. plastic
C. metal
D. sand
Reasoning: Plants need sunlight for photosynthesis.
Final Answer: A

Now answer the following.

Question:
"""
    prompt += question + "\n\nChoices:\n"
    for opt in options:
        prompt += f"{opt['label']}. {opt['text']}\n"
    prompt += "\nReasoning:\nExplain briefly.\nThen output the final answer in this format:\nFinal Answer: A/B/C/D\n"
    return prompt

# -------------------------------
# 5️⃣ Robust Answer Extraction
# -------------------------------
def extract_answer(text):
    """
    Extract final answer from model output.
    Returns single uppercase letter or 'X' if missing.
    """
    text = text.strip()
    if not text:
        return "X"

    if "Final Answer:" in text:
        ans = text.split("Final Answer:")[-1].strip()
    elif "Answer:" in text:
        ans = text.split("Answer:")[-1].strip()
    else:
        ans = text

    if len(ans) == 0:
        return "X"
    first_char = ans[0].upper()
    if first_char in ["A", "B", "C", "D"]:
        return first_char
    else:
        return "X"

# -------------------------------
# 6️⃣ Inference function (prompt_type determines which builder)
# -------------------------------
def infer(sample, tokenizer, model, prompt_type="improved"):
    if prompt_type == "original":
        prompt = build_prompt_original(sample)
    else:
        prompt = build_prompt_improved(sample)

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(
        inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=80,  # increased for reasoning in improved prompt
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id
    )
    full_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # For original prompt, the answer is after the last "Answer:"
    # For improved, we have the structured format; extract_answer works for both.
    pred = extract_answer(full_text)
    return full_text, pred

# -------------------------------
# 7️⃣ Evaluation function (runs for one prompt type)
# -------------------------------
def evaluate(samples, tokenizer, model, prompt_type, collect_reasoning=False, max_reasoning=2):
    correct = 0
    reasoning_examples = []
    for idx, sample in enumerate(tqdm(samples, desc=f"{prompt_type}")):
        text, pred = infer(sample, tokenizer, model, prompt_type)
        if pred == sample["answerKey"].upper():
            correct += 1
        elif pred == "X":
            # Optional: print warnings
            pass

        # Collect reasoning examples only for improved prompt and only first few
        if collect_reasoning and len(reasoning_examples) < max_reasoning:
            reasoning_examples.append({
                "question": sample["question"],
                "prediction": pred,
                "answerKey": sample["answerKey"].upper(),
                "reasoning_output": text
            })
    accuracy = correct / len(samples) * 100
    return accuracy, reasoning_examples

# -------------------------------
# 8️⃣ Main benchmark loop over models and prompt types
# -------------------------------
results = {}          # results[model][task][prompt] = accuracy
reasoning_samples = {} # store a few examples for the improved prompt

for label, model_name in models.items():
    print("\n" + "="*60)
    print(f"Loading Model: {label} ({model_name})")
    print("="*60)
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        torch_dtype=torch.float16
    )

    results[label] = {}
    reasoning_samples[label] = {}

    for task in tasks:
        print(f"\n--- Task: {task} ---")
        samples = task_data[task]
        if NUM_SAMPLES and NUM_SAMPLES < len(samples):
            samples = samples[:NUM_SAMPLES]

        task_results = {}
        for prompt_type in ["original", "improved"]:
            print(f"  Prompt: {prompt_type}")
            start = time.time()
            # Collect reasoning only for improved prompt
            collect = (prompt_type == "improved")
            acc, rex = evaluate(samples, tokenizer, model, prompt_type,
                                collect_reasoning=collect, max_reasoning=2)
            end = time.time()
            print(f"    Accuracy: {acc:.2f}% | Time: {end-start:.2f}s")
            task_results[prompt_type] = acc
            if collect:
                reasoning_samples[label][task] = rex

        results[label][task] = task_results

    del model
    torch.cuda.empty_cache()

# -------------------------------
# 9️⃣ Build comparison table
# -------------------------------
rows = []
for model in results:
    for task in results[model]:
        orig = results[model][task]["original"]
        impr = results[model][task]["improved"]
        diff = impr - orig
        rows.append({
            "Model": model,
            "Task": task,
            "Original (%)": round(orig, 2),
            "Improved (%)": round(impr, 2),
            "Improvement (pp)": round(diff, 2)
        })

df_results = pd.DataFrame(rows)
# Create a pivot for a cleaner view (optional)
pivot = df_results.pivot(index="Model", columns="Task", values=["Original (%)", "Improved (%)", "Improvement (pp)"])
pivot.columns = [f"{col[0]} - {col[1]}" for col in pivot.columns]
pivot = pivot.reset_index()

print("\n" + "="*80)
print("FINAL COMPARISON TABLE")
print("="*80)
print(pivot.to_string(index=False))

# Save to CSV
df_results.to_csv("benchmark_comparison.csv", index=False)
print("\nResults saved to 'benchmark_comparison.csv'")

# -------------------------------
# 10️⃣ Print sample reasoning from improved prompt
# -------------------------------
print("\n" + "="*80)
print("SAMPLE REASONING OUTPUTS (Improved Prompt)")
print("="*80)
for model in reasoning_samples:
    for task in reasoning_samples[model]:
        print(f"\n--- {model} | {task} ---")
        for ex in reasoning_samples[model][task]:
            print("Q:", ex["question"])
            print("Predicted:", ex["prediction"], "| True:", ex["answerKey"])
            # Print a snippet of the reasoning (first 500 chars)
            snippet = ex["reasoning_output"][:500] + ("..." if len(ex["reasoning_output"]) > 500 else "")
            print("Model Reasoning (excerpt):\n", snippet)
            print("-"*50)

Loading Open-LLM Benchmark dataset...


README.md: 0.00B [00:00, ?B/s]

ARC.json: 0.00B [00:00, ?B/s]

CommonsenseQA.json: 0.00B [00:00, ?B/s]

Hellaswag.json: 0.00B [00:00, ?B/s]

...

    Accuracy: 61.50% | Time: 953.40s

--- Task: PiQA ---
  Prompt: original


original: 100%|██████████| 200/200 [15:28<00:00,  4.64s/it]


    Accuracy: 72.50% | Time: 928.59s
  Prompt: improved


improved: 100%|██████████| 200/200 [16:15<00:00,  4.88s/it]


    Accuracy: 65.50% | Time: 975.30s

FINAL COMPARISON TABLE
        Model  Original (%) - CommonsenseQA  Original (%) - OpenbookQA  Original (%) - PiQA  Improved (%) - CommonsenseQA  Improved (%) - OpenbookQA  Improved (%) - PiQA  Improvement (pp) - CommonsenseQA  Improvement (pp) - OpenbookQA  Improvement (pp) - PiQA
DeepSeek-1.5B                          21.5                       36.0                 55.5                          18.5                       15.0                 38.0                              -3.0                          -21.0                    -17.5
   Qwen2.5-3B                          34.5                       39.5                 54.0                          24.5                       28.0                 53.0                             -10.0                          -11.5                     -1.0
     Qwen3-4B                          49.5                       60.5                 72.5                          37.5                       61.5           